# Exercise 02: Image classification with a CNN

## Goal

You will create and train a convolutional neural network to classify MRI scans of the brain. One class contains scans with a brain tumor and the other scans without tumor. The images are availabe in two folders. The folder ```yes``` contains the images with a tumor and the folder ```no``` those without tumor.

What you will learn in the exercise:

* how to prepare the data
  * split into train and validation data
  * use data augmentation
  * normalize the data
* Create and train a convolutional neural network
* Compute metrics to evaluate the model

## Download data

Install the dload package, that allows to download and unzip a zip-archive.

In [ ]:
!pip install git+https://github.com/regtm/dload.git

Download and unzip the data for the exercise. Change the path ``/content/`` to the path on your machine, where you want to save the data.

In [ ]:
import dload
__file__ = "."
url = 'https://github.com/MontpellierRessourcesImagerie/mri-workshop-machine-learning/releases/download/v2025-pre/data.zip'
dload.save_unzip(url, "/content/")

## Exercise

Write the code in the empty cells below, following the instructions and execute them. The cells that are already filled in, you just have to execute them (shift + enter).


The next cell switches off warnings, so that we do not get distracted by them.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

We import the python modules we will use in the exercise.

* **tensorflow:** Create and use ANNs.
* **numpy:** Vector and matrix calculations
* **classification_report:** Report metrics to evaluate the model
* **confusion_matrix:** Calculate a confusion matrix to evaluate the model
* **pyplot:** Create plots
* **Counter** We will use the counter to calculate the ratio between the sizes of the two classes, positive and negative.

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from collections import Counter
print(tf.keras.__version__)

### Exercise 2.1

We create a training dataset and a validation dataset from the images in the folders. The path must point to the top-folder, containing the subfolders with instances of the different classes.

Set the missing values for the parameters in the cell below.

* Set the path to the input directory, the directory which contains the ```yes``` and ```no``` folders.
* Let the function infer the labels from the subfolders.
* Use the binary label mode for usage with the binary-cross-entropy loss-function
* Resize the images in x and y so that all images will have the same size
* Set the validation split as a fraction, i.e. 0.2 means 20% of the images will be used for validation

In [ ]:
import os
from keras.utils import image_dataset_from_directory

path = 
training_dataset, validation_dataset = image_dataset_from_directory(
    path,
    labels= ,
    label_mode= ,
    batch_size= ,
    image_size= ,
    validation_split= ,
    subset = 'both'
    pad_to_aspect_ratio=True,
    color_mode='grayscale',
    shuffle=True,
    seed=1234,
    )

The ``image_dataset_from_directory``-function returns a [tf.data.Dataset](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) object. The Dataset contains the pairs of images and labels for each batch.

In [ ]:
ds = training_dataset.take(1).get_single_element()
print(ds[0].shape)
print(tf.transpose(ds[1]))

We display two pairs of ground truth and data.

In [ ]:
ds1 = training_dataset.take(1).get_single_element()
print(ds1[1][0].numpy())
plt.imshow(ds1[0][0], cmap='gray')
plt.show()
ds2 = training_dataset.take(1).get_single_element()
print(ds2[1][0].numpy())
plt.imshow(ds2[0][0], cmap='gray')
plt.show()

### Exercise 2.2

Apply data augmentation to the training dataset. Add horizontal random flips and random scaling to the images. 

We can use keras layers in the same way as we use them to build our model to do the data-augmentation. We need to map the data augmentation to the dataset, meaning that it will be applied when elements of the dataset are accessed. This is necessary, since the dataset is created on the fly and shuffled each time it is used. 

Use the layers:

    - tf.keras.layers.RandomFlip
    - tf.keras.layers.RandomZoom

In [ ]:
data_augmentation = tf.keras.models.Sequential(
    [
        tf.keras.layers...
        tf.keras.layers...
    ])
training_dataset = training_dataset.map(lambda x, y: (data_augmentation(x), y))

We display two pairs of ground truth and data again. This time the augmentation is applied.

In [ ]:
ds1 = training_dataset.take(1).get_single_element()
print(ds1[1][0].numpy())
plt.imshow(ds1[0][0], cmap='gray')
plt.show()
ds2 = training_dataset.take(1).get_single_element()
print(ds2[1][0].numpy())
plt.imshow(ds2[0][0], cmap='gray')
plt.show()

### Exercise 2.3

Create a CNN consisting of a convolutional part followed by a Flatten layer and a fully connected part. Display the summary of the model at the end using ``model.summary()``. The summary can help you to check if the aritechture is resonable. Check the sizes of the feature maps and the number of parameters in the fully connected part. Do not make the model too big, too many parameters can lead to overfitting.

Here are some of the layer types you might want to use:
    
    - tf.keras.layers.Rescaling(factor, input_shape=(image_height, image_width, nr_of_channels))
      - You need to set the input_shape only for the first layer
    - tf.keras.layers.Conv2D(nr_of_convolutions,
                             (filter_size_x, filter_size_y),
                             padding=padding_mode,
                             activation=activation_function)
    - tf.keras.layers.AveragePooling2D(height, width)
    - tf.keras.layers.MaxPooling2D(height, width)
    - tf.keras.layers.Flatten()
    - tf.keras.layers.Dense(number_of_units, activation=activation_function)  
    - tf.keras.layers.Dropout(rate)
    - The output unit is just another dense layer with an appropriate activation function

The model is created by passing a list of layers to ``tf.keras.models.Sequential(list_of_layers)``. Assign the result to the variable ```model```. If necessary consult the keras documentation to find out about the parameters you can use with each type of layer.

In [ ]:
model = tf.keras.models.Sequential([













])
model.summary()

### Exercise 2.4

Set the optimizer, the loss function and at least one additional metric, for example accuracy, by calling the method ``model.compile(optimizer=optimizer, loss=loss_function, metrics=list_of_metrics)``.

To create for example the adam optimizer use ``tf.keras.optimizers.Adam(learning_rate=0.001)``.

In [ ]:
additional_metric = ...
model.compile(
    ...

)

We calculate the ratio between the number of elements in the two classes. It will be used to adjust the calculation of the loss and the metrics to the imbalance.

In [ ]:
labels = list(os.walk(path))
labels = ['no' for l in labels[1][2]] + ["yes" for l in labels[2][2]]
counter = Counter(labels)
max_val = float(max(counter.values()))
class_weights = {index : max_val/num_images for index, (class_id, num_images) in enumerate(counter.items())}
print(class_weights)

### Exercise 2.5

Train the network by calling ``model.fit(...)``. The method returns the history of the loss, the validation loss (val_loss) and of the additional metrics. Set the number of epochs.

In [ ]:
history = model.fit(
    training_dataset,
    class_weight=class_weights,
    validation_data = validation_dataset,
    epochs=...,
    verbose=1,
    )

### Exercise 2.6

We plot the history of the loss and the validation loss. If the loss is getting smaller over time the model is learning. If the validation loss does not evolve similar to the loss, it means that the model is not generalizing, i.e. overfitting.

Is the model learning? Does it show signs of overfitting?

In [ ]:
plt.plot(history.history['loss'][1:])
plt.plot(history.history['val_loss'][1:])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train loss', 'validation loss'], loc='lower left')
plt.show()

In [ ]:
plt.plot(history.history[additional_metric])
plt.plot(history.history['val_' + additional_metric])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['accuracy', "val_accuracy"], loc='lower right')
plt.show()

In [ ]:
model.evaluate(validation_dataset, verbose=2)

### Exercise 2.7

We will evaluate the model on the test dataset.

Load the test dataset, the same way you loaded the training dataset before. We do not need to shuffle the data this time.

In [ ]:
test_path = 
test_dataset = image_dataset_from_directory(
    test_path,
    color_mode='grayscale',
    labels='inferred',
    label_mode='binary',
    batch_size=...,
    image_size=(256, 256),
    pad_to_aspect_ratio=True,
    shuffle=False,
    )

We run the evaluation of the model on the test-dataset.

In [ ]:
model.evaluate(test_dataset, verbose=2)

We calculate the confusion matrix and display a report of different metrics.

In [ ]:
gt = [label.numpy()[0].item()  for label in list(test_dataset)[0][1]]
Y_pred = model.predict(test_dataset)
y_pred = np.rint(Y_pred)
print('Confusion Matrix')
print(confusion_matrix(gt, y_pred))
print('Classification Report')
print(classification_report(gt, y_pred, target_names=['no', 'yes']))